In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

## MCP Server

This module exposes litesearch as a [Model Context Protocol](https://modelcontextprotocol.io) server.
Agents pull what they need at runtime via MCP tool calls rather than having context pre-stuffed.

**Tools exposed:**
- `search_documents` — hybrid FTS + vector search, returns ranked JSON results
- `get_chunk_context` — expand a result by rowid ± window for iterative deepening
- `index_document` — add new content to the store at runtime
- `get_store_stats` — count and metadata about what's indexed

**Start the server:**
```bash
litesearch-mcp --db my.db --store store --transport stdio
```

In [ ]:
#| export
from __future__ import annotations
import json
import numpy as np
from fastcore.script import call_parse, Param

In [ ]:
#| export
_db   = None   # litesearch Database, set by init_server()
_enc  = None   # FastEncode instance
_store = 'store'
_dtype = np.float16

def init_server(db_path: str,          # path to .db file (or ':memory:')
                store: str = 'store',  # table name
                model_dict = None,     # FastEncode model dict (default: embedding_gemma)
                dtype = np.float16,    # embedding dtype
                ):
    "Initialise the module-level db and encoder used by all MCP tools."
    global _db, _enc, _store, _dtype
    from litesearch import database
    from litesearch.utils import FastEncode, embedding_gemma
    _db    = database(db_path)
    _db.get_store(store)
    _enc   = FastEncode(model_dict or embedding_gemma)
    _store = store
    _dtype = dtype

In [ ]:
#| export
def _require_init():
    if _db is None: raise RuntimeError('Call init_server() before using MCP tools.')

In [ ]:
#| export
def search_documents(query: str,          # natural-language search query
                     top_k: int = 10,     # maximum number of results to return
                     store: str = None,   # table name (defaults to server store)
                     ) -> str:
    """Search indexed documents using hybrid FTS + vector search with RRF ranking.
    Returns a JSON array of {content, score, source, rank}."""
    _require_init()
    tbl = store or _store
    emb = _enc.encode_query([query])[0].ravel().tobytes()
    results = _db.search(query, emb, table_name=tbl, limit=top_k, dtype=_dtype)
    return json.dumps([{'content': r.get('content', ''), 'score': r.get('_rrf_score', 0.0),
                        'source': r.get('metadata', ''), 'rank': i}
                       for i, r in enumerate(results or [])])

In [ ]:
#| export
def get_chunk_context(rowid: int,         # rowid from a search result
                      window: int = 2,    # number of neighbouring chunks to include on each side
                      store: str = None,  # table name (defaults to server store)
                      ) -> str:
    """Expand a search result to include surrounding chunks for iterative deepening.
    Returns a JSON array of {rowid, content, metadata} ordered by rowid."""
    _require_init()
    tbl = store or _store
    rows = list(_db.q(
        f'SELECT rowid, content, metadata FROM {tbl} WHERE rowid BETWEEN ? AND ? ORDER BY rowid',
        (rowid - window, rowid + window)
    ))
    return json.dumps(rows)

In [ ]:
#| export
def index_document(content: str,          # text content to index
                   metadata: str = '',    # optional JSON string or plain string metadata
                   store: str = None,     # table name (defaults to server store)
                   ) -> str:
    """Index a new document at runtime. Embeds and stores the content.
    Returns JSON with the inserted row id."""
    _require_init()
    tbl = store or _store
    emb = _enc.encode_document([content])[0].ravel().tobytes()
    row = _db.t[tbl].insert(dict(content=content, embedding=emb, metadata=metadata))
    return json.dumps({'id': row['id'] if isinstance(row, dict) else getattr(row, 'id', None)})

In [ ]:
#| export
def get_store_stats(store: str = None  # table name (defaults to server store)
                    ) -> str:
    """Return stats about the indexed store: row count and table name.
    Returns JSON with {store, count}."""
    _require_init()
    tbl = store or _store
    count = list(_db.q(f'SELECT COUNT(*) as n FROM {tbl}'))[0]['n']
    return json.dumps({'store': tbl, 'count': count})

In [ ]:
#| export
def make_mcp_server(db_path: str,         # path to .db file
                    store: str = 'store', # table name
                    model_dict = None,    # FastEncode model dict
                    dtype = np.float16,   # embedding dtype
                    ):
    "Build and return a configured FastMCP server with all litesearch tools."
    from fastmcp import FastMCP
    init_server(db_path, store=store, model_dict=model_dict, dtype=dtype)
    mcp = FastMCP('litesearch')
    for fn in (search_documents, get_chunk_context, index_document, get_store_stats):
        mcp.tool()(fn)
    return mcp

In [ ]:
#| export
@call_parse
def run_mcp(
    db: str = ':memory:',   # path to .db file (or :memory:)
    store: str = 'store',   # table name to expose
    transport: str = 'stdio', # MCP transport: stdio or sse
):
    "Start the litesearch MCP server."
    mcp = make_mcp_server(db, store=store)
    mcp.run(transport=transport)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()